In [2]:
import os
import time
import numpy as np
import torch
from tqdm import tqdm
import pandas as pd
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from tensorboardX import SummaryWriter
from Model.SPTransformerFine import PGKPR
from Model.transformer.optimal import get_std_opt, NoamOpt
from Model.transformer.train import Batch
from Model.transformer.lossfunction import LabelSmoothing, LossCompute
from evaluation import test_bleu_rouge, normalize_txt
from dataloader import loadVocab, load_data_tensor_con

EPOCHS = 15
BATCH_SIZE = 128
VAL_BATCH_SIZE = 128
MAX_LEN = 16
LOWERCASE = True
D_MODEL = 512
tqdm_print_step = 1
MID_LOSS = True
SWITCH_INPUT = True
DOUBLE_POSITION = False


if_MID_LOSS = True
if_CON_LOSS = True

# weight
MID_WEIGHT = 1.0  # Reconstruction Loss
CON_WEIGHT = 1.0  # contrastive Loss
S2S_WEIGHT = 1.0  # s2s Loss


task_name = 'mscoco'
log_path = './grade_runs'

vocab_path = 'data/mscoco/quora.8000'

train_path = 'data/mscoco/stage_two_xy/train.csv'
train_path_xx = 'data/mscoco/stage_two_xx/train.csv'

val_path = 'data/mscoco/stage_two_xx/train.csv'
val_path_xy = 'data/mscoco/stage_two_xy/test.csv'

cur_time = time.strftime("%Y%m%d-%H.%M.%S", time.localtime())

model_save_path = 'grade_model_save/' + task_name
test_txt_save_path = 'test_out/' + task_name + '/'



class SPLoss:
    def __init__(self, vocab_size, mid_weight=0.5, con_loss_weight=0.5, con_temperature=0.1, opt=None, if_con=True, if_mid=True):
        self.mid_weight = mid_weight
        self.if_con_loss = if_con
        self.if_mid_loss = if_mid

        self.con_loss_weight = con_loss_weight
        self.con_temperature = con_temperature

        self.vocab_size = vocab_size
        self.seq2seq_criterion = nn.CrossEntropyLoss(ignore_index=0)
        self.con_loss_criterion = ContrastiveLoss(temperature=con_temperature)
        self.ce_loss = nn.CrossEntropyLoss(ignore_index=0)
        self.opt = opt

    def __call__(self, mid_output, mid_label,  # mid loss
                 sen, sen_label,  # s2s loss
                 src_encode_feature=None, tgt_encode_feature=None, src_tokes=None, tgt_with_pos_tokens=None,  # con_loss
                 train=True):
        if self.opt:
            self.opt.zero_grad()
        # mid_label = mid_index.view(-1) * mid_label.view(-1)
        mid_label = mid_label.view(-1)
        if self.if_mid_loss:
            mid_loss = self.ce_loss(mid_output.view(-1, self.vocab_size), mid_label)
        else:
            mid_loss = 0
        s2s_loss = self.seq2seq_criterion(sen.view(-1, sen.size(-1)),
                                          sen_label.contiguous().view(-1))
        if self.if_con_loss:
            con_loss = self.con_loss_criterion(src_encode_feature, tgt_encode_feature, src_tokes, tgt_with_pos_tokens)
        else:
            con_loss = 0
        total_loss = s2s_loss + self.mid_weight * mid_loss + self.con_loss_weight * con_loss
        if self.opt is not None and train:
            total_loss.backward()
            self.opt.step()

        return mid_loss, s2s_loss.data, con_loss, total_loss


class ContrastiveLoss:
    def __init__(self, temperature=0.1):
        self.temperature = temperature
        self.similarity_fun = nn.CosineSimilarity(dim=-1)

    @staticmethod
    def _sentence_embedding(encoder_out, tokens):
        # encoder_output = encoder_out.transpose(0, 1)
        encoder_output = encoder_out
        mask = (tokens != 0)  # pad: 0
        mask_ = mask.unsqueeze(-1)
        encoder_embedding = (encoder_output * mask_).sum(dim=1) / mask.float().sum(dim=1).unsqueeze(
            -1)  # [batch, hidden_size]
        return encoder_embedding

    @staticmethod
    def item(tensor):
        if hasattr(tensor, "item"):
            return tensor.item()
        if hasattr(tensor, "__getitem__"):
            return tensor[0]
        return tensor

    def __call__(self, encoder_out1, encoder_out2, src_tokens, tgt_tokens):

        encoder_embedding1 = ContrastiveLoss._sentence_embedding(encoder_out1, src_tokens)  # [batch, hidden_size]
        encoder_embedding2 = ContrastiveLoss._sentence_embedding(encoder_out2, tgt_tokens)  # [batch, hidden_size]

        batch_size = encoder_embedding2.shape[0]
        feature_dim = encoder_embedding2.shape[1]
        anchor_feature = encoder_embedding1
        contrast_feature = encoder_embedding2

        anchor_dot_contrast = self.similarity_fun(anchor_feature.expand((batch_size, batch_size, feature_dim)),
                                                  torch.transpose(
                                                      contrast_feature.expand((batch_size, batch_size, feature_dim)),
                                                      0, 1))
        loss = -nn.LogSoftmax(0)(torch.div(anchor_dot_contrast, self.temperature)).diag().sum()
        src_ntokens = ContrastiveLoss.item((src_tokens[:, 1:] != 0).int().sum().data)
        # loss = loss * src_ntokens / src_tokens.size(0)
        loss = loss / src_tokens.size(0)
        return loss



In [6]:
test_txt_save_path

'test_out/mscoco/'

In [5]:
model_save_path

'grade_model_save/mscoco'

In [8]:
PAD_TAG = '[pad]'  # 0
UNK_TAG = '[unk]'
UN_KEY_TAG = '[unkey]'
START_TAG = '<s>'
END_TAG = '</s>'
SEP_TAG = '<sep>'

In [9]:
def unkey_tager(index, sen):
    res = [UN_KEY_TAG] * len(sen)
    for i in index:
        if i != -1 and i < len(sen):
            res[i] = sen[i]
    return res

In [22]:
df = pd.read_csv("Data/mscoco/stg_two_xx/train_flatten.csv")

In [29]:
df['sen1']

0         there is a old train that is coming up the tracks
1         People sit at a table looking down at their pl...
2              a bed room a canopy and a dresser and a lamp
3             two people standing on a field with a frisbee
4                 two public transit buses on a city street
                                ...                        
464995      Some cooked nourishment that light up the day .
464996    A few goats covered in blankets standing in a ...
464997    Attendants are walking near a huge parked airp...
464998                a cake with some slices cut out of it
464999    A couple of men on horses and people on bicycl...
Name: sen1, Length: 465000, dtype: object

In [17]:
df[['question1', 'question2', 'sen1', 'sen2', 'pos1', 'pos2', 's_sen1', 's_pos1', 's_sen2', 's_pos2']]

KeyError: "['s_sen2', 's_pos2'] not in index"

In [10]:
def load_text_con(path, maxlen=25, lowercase=True, isdebug=False, switch_pos=True, reverse=False):
    df = pd.read_csv(path)

    if lowercase:
        for i in df.columns:
            if i in ['question1', 'question2', 'sen1', 'sen2', 'pos1', 'pos2', 's_sen1', 's_pos1']:
                df[i] = df[i].apply(lambda x: x.lower())
    if isdebug:
        df = df[:1]

    tgt_sen = [i.strip().split(" ")[:maxlen] for i in df.sen2.to_list()]
    labels = [[START_TAG] + x + [END_TAG] for x in tgt_sen]

    s_src_sen = [i.strip().split(" ")[:maxlen] for i in df.s_sen1.to_list()]
    s_src_pos = [i.strip().split(" ")[:maxlen] for i in df.s_pos1.to_list()]

    inputs = [[START_TAG] + x + [SEP_TAG] + y + [END_TAG] for x, y in zip(s_src_pos, s_src_sen)]

    src_sen = [i.strip().split(" ")[:maxlen] for i in df.sen1.to_list()]
    src_pos = [i.strip().split(" ")[:maxlen] for i in df.pos1.to_list()]

    label_sen = [i.strip().split(" ")[:maxlen] for i in df.sen2.to_list()]
    label_pos = [i.strip().split(" ")[:maxlen] for i in df.pos2.to_list()]

    mid_indexs = [str(i).split(" ") for i in df.s_index.to_list()]
    mid_indexs = [[int(i) for i in j if int(i) < maxlen] for j in mid_indexs]
    src_sen_unkey_targed = [unkey_tager(i, j) for i, j in zip(mid_indexs, src_sen)]

    mid_label = [[START_TAG] + x + [SEP_TAG] + y + [END_TAG] for x, y in zip(src_pos, src_sen_unkey_targed)]

    # label_with_pos = [[START_TAG] + x + [SEP_TAG] + y + [END_TAG] for x, y in zip(label_sen, label_pos)]
    label_with_pos = [[START_TAG] + y + [SEP_TAG] + x + [END_TAG] for x, y in zip(label_sen, label_pos)]

    return inputs, mid_label, labels, label_with_pos

In [ ]:
def load_data_tensor_con(path, vocabPath, maxlen=25, lowercase=True, switch_pos=True, reverse=False):
    inputs_pd, mid_label_pd, labels_pd, label_with_pos_pd = [], [], [], []
    token2idx, _ = loadVocab(vocabPath)

    inputs, mid_label, labels, label_with_pos = load_text_con(path, maxlen, lowercase=lowercase, reverse=reverse)
    padded_length = maxlen * 2 + 3
    for x, y, z, k in zip(inputs, mid_label, labels, label_with_pos):
        x = [token2idx.get(t, token2idx[UNK_TAG]) for t in x]
        y = [token2idx.get(t, token2idx[UNK_TAG]) for t in y]
        z = [token2idx.get(t, token2idx[UNK_TAG]) for t in z]
        k = [token2idx.get(t, token2idx[UNK_TAG]) for t in k]

        x = x + [0] * (padded_length - len(x))  # pad:0
        y = y + [0] * (padded_length - len(y))  # pad:0
        z = z + [0] * (padded_length - len(z))
        k = k + [0] * (padded_length - len(k))

        inputs_pd.append(x)
        mid_label_pd.append(y)
        labels_pd.append(z)
        label_with_pos_pd.append(k)
        assert len(x) == len(k)
    return torch.tensor(inputs_pd), torch.LongTensor(mid_label_pd), \
           torch.LongTensor(labels_pd), torch.LongTensor(label_with_pos_pd)

In [7]:

if not os.path.exists(model_save_path):
    os.mkdir(model_save_path)
if not os.path.exists(test_txt_save_path):
    os.mkdir(test_txt_save_path)

writer = SummaryWriter(os.path.join(log_path, task_name))
token2idx, idx2token = loadVocab(vocab_path, lowercase=LOWERCASE)
vocab_size = len(token2idx)
start_id = token2idx.get('<s>')

# data loader
# load_data_tensor return: inputs_pd, mid_label_pd, mid_indexs, labels_pd
#This function is the key part
train_data = load_data_tensor_con(train_path, vocab_path, MAX_LEN, lowercase=LOWERCASE, switch_pos=SWITCH_INPUT)
train_data_xx = load_data_tensor_con(train_path_xx, vocab_path, MAX_LEN, lowercase=LOWERCASE, switch_pos=SWITCH_INPUT)

val_data = load_data_tensor_con(val_path, vocab_path, MAX_LEN, lowercase=LOWERCASE, switch_pos=SWITCH_INPUT)
val_data_xy = load_data_tensor_con(val_path_xy, vocab_path, MAX_LEN, lowercase=LOWERCASE, switch_pos=SWITCH_INPUT)

val_df = pd.read_csv(val_path)

if LOWERCASE:
    for i in [f'question{i}' for i in range(2, 6)]:
        val_df[i] = val_df[i].apply(lambda x: x.lower())
targets = np.array(val_df[[f'question{i}' for i in range(2, 6)]]).tolist()
val_tgt_string = [normalize_txt(i, token2idx=token2idx, idx2token=idx2token, UNK_TAG='[unk]') for i in targets]
# val_tgt_string = normalize_txt(val_tgt_string, token2idx=token2idx, idx2token=idx2token, UNK_TAG='[unk]')

val_src_string = val_df['sen1'].apply(lambda x: x.lower()).to_list()  # for src_bleu, key_acc test
val_src_string = normalize_txt(val_src_string, token2idx=token2idx, idx2token=idx2token, UNK_TAG='[unk]')
val_src_string = [" ".join(i) for i in val_src_string]
val_indexes = [[int(i) for i in j.strip().split(" ")] for j in val_df.s_index.to_list()]

train_data = TensorDataset(*train_data)
train_sampler = RandomSampler(train_data)
train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=BATCH_SIZE)

train_data_xx = TensorDataset(*train_data_xx)
train_sampler_xx = RandomSampler(train_data_xx)
train_dataloader_xx = DataLoader(train_data_xx, sampler=train_sampler_xx, batch_size=BATCH_SIZE)

val_data = TensorDataset(*val_data)
val_dataloader = DataLoader(val_data, batch_size=VAL_BATCH_SIZE)

val_data_xy = TensorDataset(*val_data_xy)
val_dataloader_xy = DataLoader(val_data_xy, batch_size=VAL_BATCH_SIZE)

dataloader = {'train': train_dataloader, 'val': val_dataloader, 'train_xx': train_dataloader_xx}
num_of_batch = len(train_dataloader)

FileNotFoundError: [Errno 2] No such file or directory: 'data/mscoco/quora.8000'